In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.image-describe.liquid-bottle-fill",
        description="Describes the liquid level % of a bottle in an image.",
        worker_release="qwen3-instruct",
        text_prompt="""
          Examine the single main bottle in the image and estimate its liquid fill level as a percentage of the bottle's full height.
          Focus only on the largest and clearest foreground bottle. Ignore background bottles, shelf products, labels, logos, shadows, reflections, glare,
          printed graphics, and background objects seen through the bottle.
          Measure the fill level based on the vertical height of the liquid from the very bottom of the bottle to the very top of the cap — not by volume.
          Use this measurement even if the bottle tapers, curves, narrows at the neck, or has an unusual shape. A tapered bottle with liquid halfway up the
          bottle's height should be estimated near 50%, even if the actual volume is not half.
          For clear bottles with clear liquid, look for the visible liquid surface line, meniscus, horizontal boundary, refraction change, bubbles, ripples,
          or change in brightness or texture caused by the liquid. The background visible through the empty upper part of the bottle is not liquid — do not
          treat the entire transparent bottle body as filled just because the background is visible through it. If the liquid line is faint, estimate conservatively.
          Any percentage from 0 to 100 is a valid answer.
          Return only a single value in this exact format and nothing else: [X]%

        """,
        transform_into=TransformInto(),
       config=InferRuntimeConfig(
           max_new_tokens=5,
           image_size=640
        ),
        is_public=False
    )
]



### Create Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path


pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.image-describe.liquid-bottle-fill:latest"
   )
])


with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.jpg") # change to the path of your sample image
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
     print(json.dumps(result, indent=2))

print("Done")
